In [ ]:
from google.colab import files
uploaded = files.upload()

Saving spotify.csv to spotify.csv


## How Your Song Recommendation Project Works

This project leverages a machine learning approach to recommend songs based on their audio features. Here's a breakdown of each step:

In [ ]:
import pandas as pd
df = pd.read_csv("/content/spotify.csv")
print(df.head())

   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3           3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4           4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic          55       149610     False   
2              To Begin Again          57       210826     False   


In [ ]:
#data cleaning
df = df.dropna()
df = df.drop_duplicates(subset=['track_name'])
print(df.shape)

(73608, 21)


In [ ]:
features = [
    'danceability',
    'energy',
    'valence',
    'tempo'
]

music_data = df[features]

print(music_data.head())

   danceability  energy  valence    tempo
0         0.676  0.4610    0.715   87.917
1         0.420  0.1660    0.267   77.489
2         0.438  0.3590    0.120   76.332
3         0.266  0.0596    0.143  181.740
4         0.618  0.4430    0.167  119.949


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()


scaled_data = scaler.fit_transform(music_data)

print(scaled_data.shape)

(73608, 4)


In [ ]:
from sklearn.decomposition import PCA

# Reduce dimensions
pca = PCA(n_components=3)

reduced_data = pca.fit_transform(scaled_data)

print(reduced_data.shape)

(73608, 3)


In [ ]:
print('Original features:', features)
print('\nPrincipal Components (Loadings):')
print(pca.components_)

Original features: ['danceability', 'energy', 'valence', 'tempo']

Principal Components (Loadings):
[[ 0.34338756  0.5646991   0.74545129  0.08661574]
 [-0.29829848  0.80828635 -0.49020953  0.13185528]
 [ 0.87576666  0.08081021 -0.44503552 -0.16865897]]


In [ ]:
print('\nExplained Variance Ratio for each Principal Component:')
print(pca.explained_variance_ratio_)


Explained Variance Ratio for each Principal Component:
[0.51230488 0.29834116 0.114861  ]


In [ ]:
print(df.columns)

Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre'],
      dtype='object')


In [ ]:
"""from sklearn.metrics.pairwise import cosine_similarity
#similarity
similarity = cosine_similarity(scaled_data)
"""
from sklearn.neighbors import NearestNeighbors

# Create model
model = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

# Train model
model.fit(reduced_data)

NearestNeighbors(algorithm='brute', metric='cosine')

### 6. Similarity Model (NearestNeighbors)

**`NearestNeighbors(metric='cosine', algorithm='brute')`**: This is the core of the recommendation system. A `NearestNeighbors` model is initialized and trained on the `reduced_data`.

-   **`metric='cosine'`**: This specifies that cosine similarity will be used to measure the distance (or similarity) between songs. Cosine similarity measures the cosine of the angle between two vectors, making it effective for high-dimensional data where magnitude doesn't always matter as much as orientation.
-   **`algorithm='brute'`**: This indicates a brute-force search for neighbors, meaning it calculates distances between all data points to find the closest ones.

The model learns the relationships between the reduced features of all songs.

In [ ]:
def recommend(song_name):

    # Find song
    song = df[df['track_name'] == song_name]

    if song.empty:
        print("Song not found")
        return
    index = song.index[0]

    # Get song features
    song_features = reduced_data[index].reshape(1, -1)

    # Find nearest songs
    distances, indices = model.kneighbors(song_features, n_neighbors=6)

    print("\nRecommended Songs:\n")

    for i in indices[0][1:]:
        print(df.iloc[i]['track_name'])

In [ ]:
recommend("Believer")


Recommended Songs:

Moments
Paraíso Que Me Cerca
Kreatur der Nacht (feat. Isolation Berlin)
Tears
Lunatic (Original Mix - Slow Break) - Extended Mix


In [ ]:
recommend("To Begin Again")



Recommended Songs:

Dil Kyun Yeh Mera
Как ты красива сегодня
Another Life
Bloom
Turn Out the Lights


In summary, your project takes audio features of songs, cleans and scales them, reduces their complexity, and then uses a nearest-neighbor algorithm to find songs that are acoustically similar to a given input song. The core idea is that songs with similar audio characteristics are likely to be enjoyed by the same listener.